# ADA 3.0: TensorFlow Brain Training
Upload `BTC_USDT_1m_dataset.csv` to the root folder of this Colab session before running.


In [ ]:
!pip install pandas-ta scikit-learn tensorflow joblib


## 1. Data Loading and Feature Engineering


In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np

# Load Data
df = pd.read_csv('BTC_USDT_1m_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

print("Original Data Size:", len(df))

# Feature Engineering
df['returns'] = df['close'].pct_change()
df['volatility'] = df['returns'].rolling(window=20).std()
df['sma_20'] = ta.sma(df['close'], length=20)
df['sma_50'] = ta.sma(df['close'], length=50)

# MACD
macd = ta.macd(df['close'])
df['macd'] = macd['MACD_12_26_9']
df['macd_hist'] = macd['MACDh_12_26_9']

# RSI
df['rsi'] = ta.rsi(df['close'], length=14)

# Drop NaN values generated by rolling windows
df.dropna(inplace=True)
print("Data Size after Feature Engineering:", len(df))


## 2. Label Generation (Target)
We want ADA to predict if the price will go up by at least 0.05% in the next 3 minutes.


In [ ]:
# Shift the close price 3 minutes into the future
df['future_close'] = df['close'].shift(-3)

# Target = 1 if future close is > current close * 1.0005 (0.05% profit)
df['target'] = (df['future_close'] > df['close'] * 1.0005).astype(int)

# Drop the last few rows which now have NaN future_close
df.dropna(inplace=True)

# Check class balance
print("Target Class Balance:")
print(df['target'].value_counts(normalize=True) * 100)


## 3. Train-Test Split and Normalization


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Select features
features = ['returns', 'volatility', 'sma_20', 'sma_50', 'macd', 'macd_hist', 'rsi']
X = df[features].values
y = df['target'].values

# Chronological Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Normalize Data (CRITICAL for Neural Networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler so ADA can use it live!
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as scaler.pkl")


## 4. Build and Train TensorFlow Model


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Build a fast, lightweight Dense network
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification (0 to 1 confidence)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
print("\nStarting Training...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)


## 5. Export ADA's New Brain


In [ ]:
# Save the trained Keras model
model.save('ada_brain.keras')
print("Model saved as ada_brain.keras!")
print("Download both 'ada_brain.keras' and 'scaler.pkl' and move them to your ADA2 folder.")
